In [ ]:
import logging
import configparser
import os
import pickle

from src.data_formatting_and_upload import data_formatting
from src.image_preprocessing_module import image_preprocessing
from src.quality_assessment_and_quality_control import qa_qc
from src_knmi.table_and_cell_detection_model_knmi import table_and_cell_detection
from src_knmi.utils.transcription import transcription
from src.validation import validate

from src_knmi.utils.preprocessing import deskew, read_img, image_preprocessing
from src_knmi.utils.table_detection import table_detection
from src_knmi.table_and_cell_detection_model_knmi import table_and_cell_detection


In [ ]:
print("Setting up environment")
logger = logging.getLogger('meteosaver')
# Module 1: Configuration
# Load settings from user configurations. See configuration.ini file in this repository
config = configparser.ConfigParser()
# Adjust path to config.ini (since it's placed in the root directory)
# config_file_path = os.path.join(os.path.dirname(
    # os.path.dirname(__file__)), 'configuration.ini')
config.read('configs/base.ini')

# Get run_mode and number of processors from configuration file
run_mode = config['General']['run_mode']
num_processors = int(config['General'].get('num_processors', 1))

# Set up directories from the configuration file
full_datadir = config['Directories']['full_datadir'] # Directory to all the images/scans of the hydroclimatic data sheets
pre_QA_QC_transcribed_hydroclimate_data_dir = config['Directories']['pre_QA_QC_transcribed_hydroclimate_data_dir'] # Directory where pre-QA/QC transcribed data is stored
post_QA_QC_transcribed_hydroclimate_data_dir = config['Directories']['post_QA_QC_transcribed_hydroclimate_data_dir'] # Directory where post-QA/QC transcribed data is stored
final_refined_daily_hydroclimate_data_dir = config['Directories']['final_refined_daily_hydroclimate_data_dir'] # Directory for the final refined daily hydroclimate data (after all quality checks)
manually_transcribed_data_dir = config['Directories']['manually_transcribed_data_dir'] # Directory for manually transcribed data (used for validation)
validation_dir = config['Directories']['validation_dir'] # Directory for validation results comparing manually transcribed and the MeteoSaver transcribed data 
transient_transcription_output_dir = config['Directories']['transient_transcription_output_dir'] # Directory to store transient transcription output during processing
metadata_file_path = config['Directories']['metadata_file_path'] # Directory for all the stations metadata

# Get all folder names (Station Numbers) within full_datadir
all_stations = [folder for folder in os.listdir(full_datadir) 
                if os.path.isdir(os.path.join(full_datadir, folder))]


station = 'knmi'
pre_QA_QC_transcribed_hydroclimate_data_dir_station = os.path.join(pre_QA_QC_transcribed_hydroclimate_data_dir, station)
month_filename = file_path

print("Preprocessing...")
print(file_path)
conf_file = file_path.replace('.jpg', '.ini')
if not os.path.exists(conf_file):
    # remove trailing number from filename to get default config for country
    filename = os.path.splitext(os.path.basename(file_path))[0]
    country_year = filename.split('_')[0] + '_' + filename.split('_')[1]
    conf_file = conf_file.replace(filename, country_year)
    print(f"Conf file not found, using default for country: {conf_file}")
    config.read(conf_file)
    no_of_rows = config.getint('Cell_detection', 'no_of_rows')
    month = int(filename.split('_')[2])/2
    # get days in month based on month (not accounting for leap years)
    if month in [1, 3, 5, 7, 8, 10, 12]:
        no_of_rows -= 1 # including header
    elif month in [4, 6, 9, 11]:
        no_of_rows -= 2 # including header
    elif month == 2:
        no_of_rows -= 4 # including header
    print(f"Month: {month}, No of rows: {no_of_rows}")


else:
    config.read(conf_file)
    no_of_rows = config.getint('Cell_detection', 'no_of_rows')
no_of_columns = config.getint('Cell_detection', 'no_of_columns')
img = read_img(file_path)
config.read(conf_file)
blocksize = config.getint('Table_detection', 'blocksize')
C = config.getint('Table_detection', 'C')
table_width = config.getint('Table_detection', 'w')
table_height = config.getint('Table_detection', 'h')
table_x_offset = config.getint('Table_detection', 'x')
table_y_offset = config.getint('Table_detection', 'y')
reference_word = config['Table_detection']['reference']
min_cell_width_threshold = config.getint('Cell_detection', 'min_cell_width')
max_cell_width_threshold = config.getint('Cell_detection', 'max_cell_width')
min_cell_height_threshold = config.getint('Cell_detection', 'min_cell_height')
max_cell_height_threshold = config.getint('Cell_detection', 'max_cell_height')
no_of_columns = config.getint('Cell_detection', 'no_of_columns')

# Adjust blocksize and C as needed for different images (trial and error; if C is too high you lose parts of numbers, if too low you get too much noise)
image_in_grayscale, binarized_image, original_image = image_preprocessing(img, blocksize=blocksize, C=6, skew=True)

# Also adjust these:
size = (table_width, table_height)
table_offset=(table_x_offset, table_y_offset)
initial_offset=(1000, 500)
reference_word=reference_word
#############################
x, y, w, h, df = table_detection(binarized_image, reference_word=reference_word, size=size, 
                                initial_offset=initial_offset, table_offset=table_offset)

if x is not None:
    table_original_image = original_image[y:y + h, x:x + w]
    image_in_grayscale, binarized_image, original_image = image_preprocessing(table_original_image, cutoff_hsv=125, skew=False, blur=True, blocksize=15, C=3)
    
    table_img_bin = binarized_image
    table_original_image = original_image
    full_detected_table_with_labels = binarized_image
    cv2.namedWindow('custom window', cv2.WINDOW_KEEPRATIO)
    cv2.imshow('custom window', table_img_bin)
    cv2.resizeWindow('custom window', 1080, 720)
    cv2.waitKey()
    cv2.destroyAllWindows()
else:
    df.head()


print("Running Table and Cell Detection Model...")
detected_table_and_cells = table_and_cell_detection(image_in_grayscale, binarized_image, original_image, station, month_filename, transient_transcription_output_dir,
                                            clip_up = int(config['TableAndCellDetection']['clip_up']), 
                                            clip_down = int(config['TableAndCellDetection']['clip_down']),
                                            clip_left = int(config['TableAndCellDetection']['clip_left']),
                                            clip_right = int(config['TableAndCellDetection']['clip_right']),
                                            max_table_width = int(config['TableAndCellDetection']['max_table_width']),
                                            max_table_height = int(config['TableAndCellDetection']['max_table_height']),
                                            min_cell_width_threshold=min_cell_width_threshold,
                                            max_cell_width_threshold=max_cell_width_threshold,
                                            min_cell_height_threshold=min_cell_height_threshold,
                                            max_cell_height_threshold=max_cell_height_threshold,
                                            space_height_threshold=int(config['TableAndCellDetection']['space_height_threshold']), 
                                            space_width_threshold=int(config['TableAndCellDetection']['space_width_threshold']), 
                                            max_cell_height_per_box=int(config['TableAndCellDetection']['max_cell_height_per_box']), 
                                            no_of_rows=no_of_rows,
                                            no_of_columns=no_of_columns)

